In [85]:
import torch
import pickle
import argparse
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
import re
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from terrarium.models import get_model, load_model
from terrarium.dataloaders import get_dataloader

In [86]:
model_name = 'cat_v1'
ckpt_path = 'out/cat_v1/7M/7M_1.0/ckpt.pt'
data_dir = 'data/v_32768'
dl_name = 'cat_inference'
batch_size = 10
max_seq_len = 20
total_sequences = 120
data_prefix = 'val'
summary_only = False
out_dir = './'
device = 'cpu'

In [87]:
model = load_model(model_name,ckpt_path,device_override='cpu').to(device)
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False) # always first to cpu

In [88]:
with open(f"{data_dir}/tok.pkl",'rb') as f:
    tok = pickle.load(f)
sos_id = tok.encode('<|sos|>')
dl = get_dataloader(dl_name,batch_size,max_seq_len,f"{data_dir}/{data_prefix}_token_ids.bin",f"{data_dir}/{data_prefix}_attr_ids.bin",sos_id,device)

In [89]:
# check if dataset has enough sequences
if len(dl.split_locs)-1 < total_sequences:
    total_sequences = len(dl.split_locs)-1
    print(f'Warning: requested total_sequences reduced to {total_sequences} due to data size.')
n_batches = total_sequences // batch_size

In [90]:
# create the holder for results
results = {
        'true_token': [],
        'sampled_token': [],
        'top1_token': [],
        'cross_entropy_loss_token': [],
        'sampled_attr': [],
        'top1_attr': [],
        'cross_entropy_loss_attr': []
}

In [91]:
x,y,attr = dl.next_batch()
x

[tensor([ 257,  279,  258,  536,  622, 2891,   34,  259,  905,  515,   45,  470,
          360,  304, 8853, 8001,   34,  260]),
 tensor([  257,   298,   258,  2006,   622,  3020,   259,   934,  7974,   566,
           798, 13168,  2748,  7317,  7681,   260]),
 tensor([  257,   279,   258,  3897, 12483,    33,    51,    48,    52,   259,
          3897,   388, 12483,   513,   920,    47,   260]),
 tensor([ 257,  279,  258,  755,  657,  259,  577,  900,  347,  470,  358, 2635,
         2367,  349,  578, 2935, 8389, 2157,   45,  260]),
 tensor([ 257,  281,  258,  905,  338, 1241,  259,  905,  338, 1241,  260]),
 tensor([ 257,  295,  258,  755,  657,  259, 1087,  260]),
 tensor([ 257,  292,  258,  755,  657,  259, 1404,  604,  324, 1098,  260]),
 tensor([ 257,  285,  258, 1891, 2616,  259, 2224,  508,  604,  382, 1877,  358,
          642,  260]),
 tensor([ 257,  287,  258, 3339, 1831,  349,  427,  102,  332, 2702,  259,  536,
          638,   34,  260]),
 tensor([ 257,  285,  258, 1105,  

In [92]:
attrs = torch.tensor([attri[0].item() for attri in attr]).to(device)
attrs

tensor([5, 5, 5, 5, 5, 5, 5, 1, 5, 5])

In [93]:
# assemble x and y from the return sequences t.
x_pad = torch.zeros((len(x), max_seq_len), dtype=torch.long)
y_pad = torch.zeros((len(y), max_seq_len), dtype=torch.long)
seq_lengths = torch.zeros((len(x),), dtype=torch.long)
for i, (xi, yi) in enumerate(zip(x, y)):
    L = len(xi)
    seq_lengths[i] = L
    x_pad[i, :L] = xi
    y_pad[i, :L] = yi
x_pad

tensor([[  257,   279,   258,   536,   622,  2891,    34,   259,   905,   515,
            45,   470,   360,   304,  8853,  8001,    34,   260,     0,     0],
        [  257,   298,   258,  2006,   622,  3020,   259,   934,  7974,   566,
           798, 13168,  2748,  7317,  7681,   260,     0,     0,     0,     0],
        [  257,   279,   258,  3897, 12483,    33,    51,    48,    52,   259,
          3897,   388, 12483,   513,   920,    47,   260,     0,     0,     0],
        [  257,   279,   258,   755,   657,   259,   577,   900,   347,   470,
           358,  2635,  2367,   349,   578,  2935,  8389,  2157,    45,   260],
        [  257,   281,   258,   905,   338,  1241,   259,   905,   338,  1241,
           260,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  257,   295,   258,   755,   657,   259,  1087,   260,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  257,   292,   258,   755,   657,   

In [94]:
with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.bfloat16):
    logits,attr_logits,_,_ = model.forward(x_pad,targets=y_pad) # (B,T,V) and (B,T,O)

logits.shape

torch.Size([10, 20, 32786])

In [95]:
attr_logits.shape

torch.Size([10, 20, 6])

In [96]:
B,T,V = logits.shape

In [97]:
rows_to_use = torch.arange(B, device=device)
rows_to_use

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [98]:
token_probs = torch.softmax(logits,dim=-1)
token_probs.shape

torch.Size([10, 20, 32786])

In [99]:
token_probs.view(-1,V)

tensor([[9.2655e-12, 1.2676e-11, 6.7928e-12,  ..., 1.0516e-11, 8.7539e-12,
         1.3529e-11],
        [1.1990e-13, 1.1280e-13, 1.1280e-13,  ..., 1.2790e-13, 8.2601e-14,
         2.1050e-13],
        [8.2946e-10, 9.3860e-10, 9.6770e-10,  ..., 9.9681e-10, 8.8039e-10,
         6.4392e-10],
        ...,
        [1.8481e-09, 1.9645e-09, 9.2768e-10,  ..., 2.3720e-09, 1.0550e-09,
         1.9645e-09],
        [1.5061e-09, 2.0518e-09, 8.0399e-10,  ..., 2.1828e-09, 9.1313e-10,
         1.6007e-09],
        [1.8263e-09, 2.3429e-09, 1.1059e-09,  ..., 2.6484e-09, 1.1787e-09,
         1.9354e-09]], dtype=torch.bfloat16)

In [100]:
torch.multinomial(token_probs.view(-1,V),num_samples=1)

tensor([[  300],
        [  258],
        [   74],
        [ 2238],
        [ 1582],
        [  382],
        [  259],
        [ 2959],
        [  310],
        [  747],
        [ 3475],
        [  668],
        [  388],
        [14336],
        [ 1384],
        [ 3301],
        [   33],
        [  266],
        [  731],
        [ 1521],
        [  277],
        [  258],
        [  771],
        [  328],
        [  701],
        [  259],
        [ 2006],
        [ 1989],
        [  309],
        [  371],
        [ 2086],
        [ 5306],
        [  337],
        [ 3566],
        [  430],
        [  262],
        [ 2742],
        [  400],
        [10020],
        [15477],
        [  284],
        [  258],
        [25089],
        [  388],
        [ 2633],
        [   57],
        [   45],
        [   53],
        [ 1270],
        [16416],
        [  347],
        [ 3032],
        [   33],
        [ 1058],
        [ 4601],
        [  405],
        [  263],
        [  404],
        [  537

In [101]:
sampled_token = torch.multinomial(token_probs.view(-1,V),num_samples=1).view(B,T) # B,T
# sample over V dimension, sampled_token should look similar to y_pad
print(x_pad)
print(y_pad)
sampled_token

tensor([[  257,   279,   258,   536,   622,  2891,    34,   259,   905,   515,
            45,   470,   360,   304,  8853,  8001,    34,   260,     0,     0],
        [  257,   298,   258,  2006,   622,  3020,   259,   934,  7974,   566,
           798, 13168,  2748,  7317,  7681,   260,     0,     0,     0,     0],
        [  257,   279,   258,  3897, 12483,    33,    51,    48,    52,   259,
          3897,   388, 12483,   513,   920,    47,   260,     0,     0,     0],
        [  257,   279,   258,   755,   657,   259,   577,   900,   347,   470,
           358,  2635,  2367,   349,   578,  2935,  8389,  2157,    45,   260],
        [  257,   281,   258,   905,   338,  1241,   259,   905,   338,  1241,
           260,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  257,   295,   258,   755,   657,   259,  1087,   260,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  257,   292,   258,   755,   657,   

tensor([[  285,   258,   536,   732,  2482,   259,   259,    74, 14564,  2891,
           566,  2798,   310,  4934,    45,    45,   260,   265,  8559,  9731],
        [  292,   258,  2185,   399,  3320,   259,    84,  1429,  1170,   515,
          6200,   433,  7317,    47,    47,   266,   834,   388,  6678,  2017],
        [  275,   258,   761,    33,  2261,  2303, 14007,  6375,  3421,    57,
           890, 28910,    33,   604,   326,   260,   262,  1086,  1119,   984],
        [  276,   258,  4673,   657,   259,   536,   347,  8986,   892,   592,
          3474,   310,  1001,   388,  1227,   384,  1174,   358,   605,   266],
        [  286,   258,   536,   489,    45,   259,    74,  1072,    34,  4839,
           266,    34,   816,   571,  2888,   347,  2902,   537,  6001, 25994],
        [  284,   258, 28339,   657,   259,  3339,  2624,   266, 21339,   343,
          1582,   324,   932,  2372, 21142,  4798,   977, 15890,  4781, 10263],
        [  297,   258,  2716,   657,   259,   

In [102]:
torch.argmax(token_probs,dim=-1) # B,T

tensor([[  285,   258,   536,   575,  3766,   259,   259,    74,   371,  2891,
           566,   668,   388,    33,    47,    47,   260,   266,   347,   347],
        [  285,   258,   536,   328,  2099,   259,    74,  1429,   566,   338,
          3020,   328,  7317,    47,    47,   266,   433,   433,   347,   347],
        [  285,   258,   536,  1226,  1226,    51,    47,    51,    47,    74,
          1226,    33,  1226,  1058,  1275,    33,   266,   347,   347,   347],
        [  285,   258,   536,   657,   259,   536,   347,   347,   304,    47,
           310,   388,    47,   310,  1065,  6000,  1174,    47,   328,   266],
        [  285,   258,   536,   338,   259,   259,    74,   371,    34,   260,
           266,   347,   347,   347,   347,   347,   347,   347,   347,   347],
        [  285,   258,   536,   657,   259,   536,   260,   266,   260,   260,
           260,   260,   391,   260,   260,   260,   260,   391,   391,   260],
        [  285,   258,   536,   657,   259,   

In [103]:
y_pad.shape

torch.Size([10, 20])

In [104]:
y_pad.unsqueeze(-1).shape

torch.Size([10, 20, 1])

In [105]:
token_probs.shape

torch.Size([10, 20, 32786])

In [106]:
torch.gather(token_probs,2,y_pad.unsqueeze(-1)).clamp_min(1e-9).shape

torch.Size([10, 20, 1])

In [107]:
-torch.log(torch.gather(token_probs,2,y_pad.unsqueeze(-1)).clamp_min(1e-9)).squeeze(-1) # B,T

tensor([[2.5938e+00, -0.0000e+00, 2.3594e+00, 5.1562e+00, 6.2188e+00, 2.6094e+00,
         9.8633e-02, 3.2031e+00, 1.9141e+00, 4.9375e+00, 2.5938e+00, 5.1250e+00,
         4.0625e+00, 5.4688e+00, 5.3438e+00, 4.3438e+00, 7.0312e-01, 9.8633e-02,
         1.9875e+01, 1.9750e+01],
        [3.5938e+00, -0.0000e+00, 4.6875e+00, 4.0625e+00, 5.8125e+00, 7.6953e-01,
         2.5156e+00, 5.2188e+00, 8.2031e-01, 5.3125e+00, 7.0000e+00, 5.7500e+00,
         1.4609e+00, 8.6875e+00, 2.8594e+00, 3.7500e-01, 1.9875e+01, 2.0000e+01,
         2.0000e+01, 2.0000e+01],
        [2.5938e+00, -0.0000e+00, 6.7188e+00, 1.0125e+01, 3.1875e+00, 2.2656e+00,
         4.3750e+00, 2.2188e+00, 2.8906e+00, 6.7188e+00, 2.6875e+00, 4.7188e+00,
         5.0625e+00, 2.4688e+00, 2.2969e+00, 2.1875e+00, 3.4180e-01, 2.0250e+01,
         2.0375e+01, 2.0250e+01],
        [2.5938e+00, -0.0000e+00, 2.5312e+00, 3.9062e-03, 3.9062e-03, 3.7188e+00,
         3.7812e+00, 8.4766e-01, 2.6250e+00, 2.0625e+00, 5.5625e+00, 4.1875e+00,
   

In [108]:
seq_lengths.shape

torch.Size([10])

In [109]:
len(y[0])

18

In [110]:
len(y_pad[0])

20

In [111]:
(seq_lengths - 1).clamp_min(0)  # B, # original sequence lengths before padding

tensor([17, 15, 16, 19, 10,  7, 10, 13, 14, 15])

In [112]:
B,T,O = attr_logits.shape
cond_prob_obs = torch.softmax(attr_logits,dim=-1)

In [113]:
y_pad[0]

tensor([ 279,  258,  536,  622, 2891,   34,  259,  905,  515,   45,  470,  360,
         304, 8853, 8001,   34,  260,  266,    0,    0])

In [114]:
y[0]

tensor([ 279,  258,  536,  622, 2891,   34,  259,  905,  515,   45,  470,  360,
         304, 8853, 8001,   34,  260,  266])

In [115]:
(seq_lengths - 2).clamp_min(0) # n-1

tensor([16, 14, 15, 18,  9,  6,  9, 12, 13, 14])

In [116]:
cond_prob_obs.view(-1,O).shape # torch.multinomial across attribute dimension

torch.Size([200, 6])

In [117]:
torch.multinomial(cond_prob_obs.view(-1,O),num_samples=1).view(B,T)

tensor([[5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0],
        [5, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 4, 4, 5, 3, 0, 1, 2],
        [5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 2, 0, 1],
        [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
        [5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 1, 0, 0, 0, 1, 0, 4, 4],
        [5, 5, 5, 5, 5, 5, 5, 5, 0, 1, 4, 2, 4, 4, 1, 3, 1, 3, 4, 0],
        [3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 1, 2, 3, 4, 2, 1, 0],
        [5, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 3, 1, 2, 4, 4],
        [5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 0, 1, 2, 1],
        [3, 5, 4, 4, 5, 4, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 2, 3, 3, 1]])

In [118]:
cond_prob_obs.shape

torch.Size([10, 20, 6])

In [119]:
y_pad.shape

torch.Size([10, 20])

In [120]:
y_pad.unsqueeze(-1)

tensor([[[  279],
         [  258],
         [  536],
         [  622],
         [ 2891],
         [   34],
         [  259],
         [  905],
         [  515],
         [   45],
         [  470],
         [  360],
         [  304],
         [ 8853],
         [ 8001],
         [   34],
         [  260],
         [  266],
         [    0],
         [    0]],

        [[  298],
         [  258],
         [ 2006],
         [  622],
         [ 3020],
         [  259],
         [  934],
         [ 7974],
         [  566],
         [  798],
         [13168],
         [ 2748],
         [ 7317],
         [ 7681],
         [  260],
         [  266],
         [    0],
         [    0],
         [    0],
         [    0]],

        [[  279],
         [  258],
         [ 3897],
         [12483],
         [   33],
         [   51],
         [   48],
         [   52],
         [  259],
         [ 3897],
         [  388],
         [12483],
         [  513],
         [  920],
         [   47],
      

In [121]:
torch.gather(cond_prob_obs,2,y_pad.unsqueeze(-1))

RuntimeError: index 279 is out of bounds for dimension 2 with size 6

In [122]:
attr_token_ids = torch.tensor([261, 262, 263, 264, 265, 266], device=y_pad.device)

In [123]:
y_attr = (y_pad[..., None] == attr_token_ids).long().argmax(dim=-1)
has_attr = (y_pad[..., None] == attr_token_ids).any(dim=-1)

In [124]:
idx = y_attr.clamp_min(0)
nll_attr = -torch.log(torch.gather(cond_prob_obs, 2, idx.unsqueeze(-1)).clamp_min(1e-9)).squeeze(-1)
nll_attr = nll_attr * has_attr

In [125]:
idx

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0]])

In [126]:
nll_attr.shape

torch.Size([10, 20])

In [127]:
attrs = torch.tensor([attri[0].item() for attri in attr]).to(device)
attrs

tensor([5, 5, 5, 5, 5, 5, 5, 1, 5, 5])

In [ ]:
cond_prob_obs[
    torch.arange(B, device=cond_prob_obs.device),
    last_idx,
    :
].gather(1, attrs.unsqueeze(1)).squeeze(1)

In [ ]:
cond_prob_obs[

In [128]:
# --- run the inference over batches --- #
for b in tqdm(range(n_batches)):
    x,y,attr = dl.next_batch()
    attrs = torch.tensor([attri[0].item() for attri in attr]).to(device)
    
    # pad and assemble x and y
    x_pad = torch.zeros((len(x), max_seq_len), dtype=torch.long)
    y_pad = torch.zeros((len(y), max_seq_len), dtype=torch.long)
    seq_lengths = torch.zeros((len(x),), dtype=torch.long)
    for i, (xi, yi) in enumerate(zip(x, y)):
        L = len(xi)
        seq_lengths[i] = L
        x_pad[i, :L] = xi
        y_pad[i, :L] = yi
        
    x_pad,y_pad,seq_lengths = x_pad.to(device),y_pad.to(device),seq_lengths.to(device)
    
    # forward pass
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        logits,attr_logits,_,_ = model.forward(x_pad,targets=y_pad) # (B,T,V) and (B,T,A)
    B,T,V = logits.shape
    rows_to_use = torch.arange(B, device=device)
    token_probs = torch.softmax(logits,dim=-1)

    sampled_token = torch.multinomial(token_probs.view(-1,V),num_samples=1).view(B,T) # B,T
    top1_token = torch.argmax(token_probs,dim=-1) # B,T
    nll_token = -torch.log(torch.gather(token_probs,2,y_pad.unsqueeze(-1)).clamp_min(1e-9)).squeeze(-1) # B,T

    last_idx = (seq_lengths - 1).clamp_min(0)  # B,
    true_last_token = y_pad[rows_to_use, last_idx]
    sampled_last_token = sampled_token[rows_to_use, last_idx]
    top1_last_token = top1_token[rows_to_use, last_idx]
    nll_last_token = nll_token[rows_to_use, last_idx]
    
    decode_num = lambda t: int(re.search(r"<\|\*_(\d+)\|>", tok.decode([int(t)])).group(1))
    
    results['true_token'].append([decode_num(t) for t in true_last_token.tolist()])
    results['sampled_token'].append([decode_num(t) for t in sampled_last_token.tolist()])
    results['top1_token'].append([decode_num(t) for t in top1_last_token.tolist()])
    results['cross_entropy_loss_token'].append(nll_last_token.tolist())

    B,T,A = attr_logits.shape
    cond_prob_obs = torch.softmax(attr_logits,dim=-1)

    n_minus_1_idx = (seq_lengths - 2).clamp_min(0)  # B,
    sampled_attr = torch.multinomial(cond_prob_obs.view(-1,A),num_samples=1).view(B,T) # B,T
    top1_attr = torch.argmax(cond_prob_obs,dim=-1) # B,T
    nll_attr = -torch.log(cond_prob_obs.clamp_min(1e-9)) # B,T

    results['sampled_attr'].append(sampled_attr[rows_to_use, n_minus_1_idx].tolist())
    results['top1_attr'].append(top1_attr[rows_to_use, n_minus_1_idx].tolist())
    results['cross_entropy_loss_attr'].append(nll_attr[rows_to_use, n_minus_1_idx, attrs[rows_to_use]].tolist())

100%|██████████| 12/12 [00:02<00:00,  4.91it/s]


In [129]:
# ---- data post processing ---- #
results['true_token'] = [item for sublist in results['true_token'] for item in sublist]
results['sampled_token'] = [item for sublist in results['sampled_token'] for item in sublist]
results['top1_token'] = [item for sublist in results['top1_token'] for item in sublist]
results['cross_entropy_loss_token'] = [item for sublist in results['cross_entropy_loss_token'] for item in sublist]
results['sampled_attr'] = [item for sublist in results['sampled_attr'] for item in sublist]
results['top1_attr'] = [item for sublist in results['top1_attr'] for item in sublist]
results['cross_entropy_loss_attr'] = [item for sublist in results['cross_entropy_loss_attr'] for item in sublist]

In [130]:
print(results)

{'true_token': [5, 5, 3, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, 3, 5, 5, 5, 3, 5, 5, 4, 5, 1, 4, 5, 5, 5, 5, 5, 3, 5, 1, 5, 5, 5, 4, 4, 5, 5, 5, 5, 1, 5, 5, 5, 5, 5, 4, 4, 5, 5, 4, 5, 2, 5, 1, 4, 5, 1, 5, 5, 4, 3, 5, 5, 5, 5, 5, 5, 4, 4, 4, 1, 5, 1, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 4, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 4, 5, 5, 5, 4, 5, 5, 3, 5, 3, 5], 'sampled_token': [5, 4, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 3, 5, 5, 5, 4, 5, 4, 5, 5, 2, 4, 5, 5, 5, 5, 5, 4, 5, 1, 5, 2, 4, 4, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 3, 1, 5, 5, 4, 5, 4, 5, 3, 3, 4, 2, 5, 5, 1, 2, 5, 5, 5, 5, 5, 5, 3, 5, 4, 1, 5, 1, 4, 4, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 4, 5, 3, 5, 5, 4, 5, 5, 5, 1, 5], 'top1_token': [5, 5, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, 3, 5, 5, 5, 4, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 1, 5, 5, 5, 4, 5, 5, 5, 5, 5, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 1, 4, 5, 1, 5, 5, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5

In [131]:
final_df = pd.DataFrame({
    'true_token': results['true_token'],
    'sampled_token': results['sampled_token'],
    'top1_token': results['top1_token'],
    'cross_entropy_loss_token': results['cross_entropy_loss_token'],
    'sampled_attr': results['sampled_attr'],
    'top1_attr': results['top1_attr'],
    'cross_entropy_loss_attr': results['cross_entropy_loss_attr']
})

In [132]:
final_df

,true_token,sampled_token,top1_token,cross_entropy_loss_token,sampled_attr,top1_attr,cross_entropy_loss_attr
0,5,5,5,0.039795,5,5,0.040232
1,5,4,5,0.271484,5,5,0.001311
2,3,5,1,1.937500,1,5,2.693024
3,5,5,5,0.019775,5,5,0.009200
4,5,5,5,0.151367,5,5,0.152690
...,...,...,...,...,...,...,...
115,5,5,5,0.267578,5,5,0.244311
116,3,5,1,2.218750,3,3,0.017821
117,5,5,5,0.064453,5,5,0.000462
118,3,1,3,1.039062,3,3,0.090524


In [133]:
# concatenate all dataframes
# final_df.to_csv('./out/online_classification_results.csv', index=False) # not sure want to save, could be massive
if not summary_only:
    final_df.to_csv(os.path.join(out_dir,'online_classification_results.csv'), index=False)

In [135]:
# summary by true attr
df_stratified_summary = (
    final_df
    .groupby(['true_token'])
    .agg(
        n_attrs=('true_token', 'count'),
        sampled_accuracy_token=('sampled_token', lambda x: np.mean(x == final_df.loc[x.index, 'true_token'])),
        top1_accuracy_token=('top1_token', lambda x: np.mean(x == final_df.loc[x.index, 'true_token'])),
        cross_entropy_mean_token=('cross_entropy_loss_token', 'mean'),
        sampled_accuracy_attr=('sampled_attr', lambda x: np.mean(x == final_df.loc[x.index, 'true_token'])),
        top1_accuracy_attr=('top1_attr', lambda x: np.mean(x == final_df.loc[x.index, 'true_token'])),
        cross_entropy_mean_attr=('cross_entropy_loss_attr', 'mean')
    )
    .reset_index()  # make true_token into columns again
)

In [136]:
df_stratified_summary.to_csv(os.path.join(out_dir,'online_classification_stratified_summary.csv'),index=False)

In [137]:
df_stratified_summary

,true_token,n_attrs,sampled_accuracy_token,top1_accuracy_token,cross_entropy_mean_token,sampled_accuracy_attr,top1_accuracy_attr,cross_entropy_mean_attr
0,1,7,0.428571,0.857143,0.770089,0.714286,0.857143,0.680699
1,2,1,0.000000,0.000000,2.734375,1.000000,1.000000,0.008626
2,3,8,0.125000,0.500000,1.341797,0.750000,0.750000,0.725227
3,4,20,0.450000,0.350000,1.527734,0.700000,0.550000,1.218529
4,5,84,0.880952,1.000000,0.182370,0.869048,0.964286,0.189391
